# MELD Emotion Detection: LSTM vs DistilBERT vs Zero-Shot LLM

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
!git clone https://github.com/declare-lab/MELD.git

Cloning into 'MELD'...
remote: Enumerating objects: 496, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 496 (delta 12), reused 11 (delta 5), pack-reused 475 (from 1)
Receiving objects: 100% (496/496), 8.12 MiB | 22.54 MiB/s, done.
Resolving deltas: 100% (260/260), done.


In [ ]:
import pandas as pd

train_df = pd.read_csv('MELD/data/MELD/train_sent_emo.csv')
test_df  = pd.read_csv('MELD/data/MELD/test_sent_emo.csv')

X_train = train_df['Utterance'].astype(str).tolist()
y_train = train_df['Emotion'].tolist()

X_test = test_df['Utterance'].astype(str).tolist()
y_test = test_df['Emotion'].tolist()

print(len(X_train), len(X_test))

9989 2610


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)
num_classes = len(le.classes_)
print(le.classes_)

['anger' 'disgust' 'fear' 'joy' 'neutral' 'sadness' 'surprise']


## LSTM

In [ ]:
import re
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9' ]", " ", text)
    return text.split()

counter = Counter()
for sent in X_train:
    counter.update(simple_tokenize(sent))

VOCAB_SIZE = 8000
most_common = counter.most_common(VOCAB_SIZE - 2)
word2idx = {word: idx + 2 for idx, (word, _) in enumerate(most_common)}
word2idx["<pad>"] = 0
word2idx["<unk>"] = 1

MAX_LEN = 30

def encode(text):
    tokens = simple_tokenize(text)
    ids = [word2idx.get(tok, 1) for tok in tokens[:MAX_LEN]]
    ids = ids + [0] * (MAX_LEN - len(ids))
    return ids

X_train_ids = torch.tensor([encode(t) for t in X_train], dtype=torch.long)
X_test_ids  = torch.tensor([encode(t) for t in X_test], dtype=torch.long)
y_train_t   = torch.tensor(y_train_enc, dtype=torch.long)
y_test_t    = torch.tensor(y_test_enc, dtype=torch.long)

print(X_train_ids.shape, X_test_ids.shape)

torch.Size([9989, 30]) torch.Size([2610, 30])


In [ ]:
!pip install -q gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 51.9 MB/s eta 0:00:00


In [ ]:
import gensim.downloader as api
import numpy as np

glove_vectors = api.load("glove-wiki-gigaword-100")

embed_dim = 100
embedding_matrix = np.zeros((len(word2idx), embed_dim))
for word, idx in word2idx.items():
    if word in glove_vectors:
        embedding_matrix[idx] = glove_vectors[word]
    else:
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embed_dim,))
embedding_matrix[0] = np.zeros(embed_dim)

embedding_tensor = torch.tensor(embedding_matrix, dtype=torch.float)

[==================================================] 100.0% 128.1/128.1MB downloaded


In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=256, num_classes=7):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        combined = torch.cat((hidden[-2], hidden[-1]), dim=1)
        out = self.dropout(combined)
        return self.fc(out)

device = "cuda" if torch.cuda.is_available() else "cpu"
lstm_model = LSTMClassifier(embedding_tensor, num_classes=num_classes).to(device)
print(lstm_model)
print(device)

LSTMClassifier(
  (embedding): Embedding(5536, 100, padding_idx=0)
  (lstm): LSTM(100, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.4, inplace=False)
  (fc): Linear(in_features=512, out_features=7, bias=True)
)
cuda


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_enc),
    y=y_train_enc
)
soft_class_weights = np.power(class_weights, 0.5)
class_weights_t = torch.tensor(soft_class_weights, dtype=torch.float).to(device)

In [ ]:
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(SeqDataset(X_train_ids, y_train_t), batch_size=64, shuffle=True)

criterion = nn.CrossEntropyLoss(weight=class_weights_t)
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

EPOCHS = 15
for epoch in range(EPOCHS):
    lstm_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = lstm_model(xb)
        loss = criterion(out, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), max_norm=5)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(epoch + 1, total_loss / len(train_loader))

1 1.767887722155091
2 1.6425370714467042
3 1.5485969144067946
4 1.4591856192631327
5 1.3581831436248342
6 1.2195980833594207
7 1.1479561587048184
8 1.0668258556894437
9 0.9965412089019824
10 0.9272809017236066
11 0.8168882864296056
12 0.7734898879269886
13 0.7174481014916851
14 0.6643499798455815
15 0.6385797786105211


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

lstm_model.eval()
with torch.no_grad():
    preds = lstm_model(X_test_ids.to(device))
    preds = torch.argmax(preds, dim=1).cpu().numpy()

lstm_accuracy = accuracy_score(y_test_enc, preds)
print(lstm_accuracy)
print(classification_report(y_test_enc, preds, target_names=le.classes_))

0.4590038314176245
              precision    recall  f1-score   support

       anger       0.30      0.17      0.22       345
     disgust       0.11      0.07      0.09        68
        fear       0.05      0.08      0.06        50
         joy       0.42      0.41      0.41       402
     neutral       0.61      0.62      0.61      1256
     sadness       0.20      0.23      0.21       208
    surprise       0.38      0.50      0.43       281

    accuracy                           0.46      2610
   macro avg       0.29      0.30      0.29      2610
weighted avg       0.46      0.46      0.45      2610



## DistilBERT

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_classes)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
class BertDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset_bert = BertDataset(X_train, y_train_enc, tokenizer)
test_dataset_bert  = BertDataset(X_test, y_test_enc, tokenizer)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_enc),
    y=y_train_enc
)
soft_class_weights = np.power(class_weights, 0.5)
bert_class_weights = torch.tensor(soft_class_weights, dtype=torch.float)

In [ ]:
from torch.nn import CrossEntropyLoss

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = CrossEntropyLoss(weight=bert_class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

training_args = TrainingArguments(
    output_dir="./bert_out",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    logging_steps=50,
    save_strategy="no",
    report_to="none",
)

trainer = WeightedTrainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset_bert,
    eval_dataset=test_dataset_bert,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.320045,1.338466,0.634483
2,1.297733,1.315421,0.631418


TrainOutput(global_step=1250, training_loss=1.368704965209961, metrics={'train_runtime': 116.4419, 'train_samples_per_second': 171.57, 'train_steps_per_second': 10.735, 'total_flos': 330833708411136.0, 'train_loss': 1.368704965209961, 'epoch': 2.0})

In [ ]:
bert_results = trainer.evaluate()
bert_accuracy = bert_results["eval_accuracy"]
print(bert_accuracy)

bert_preds = trainer.predict(test_dataset_bert)
bert_pred_labels = np.argmax(bert_preds.predictions, axis=1)
print(classification_report(y_test_enc, bert_pred_labels, target_names=le.classes_))

Training Loss,Validation Loss,Epoch,Accuracy
1.297733,1.315421,2,0.631418


0.6314176245210728


              precision    recall  f1-score   support

       anger       0.46      0.46      0.46       345
     disgust       0.22      0.07      0.11        68
        fear       0.17      0.06      0.09        50
         joy       0.58      0.58      0.58       402
     neutral       0.78      0.80      0.79      1256
     sadness       0.35      0.30      0.32       208
    surprise       0.51      0.66      0.57       281

    accuracy                           0.63      2610
   macro avg       0.44      0.42      0.42      2610
weighted avg       0.62      0.63      0.62      2610



## Zero-shot LLM

In [ ]:
from transformers import pipeline

zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1
)

candidate_labels = list(le.classes_)

SAMPLE_SIZE = 200
sample_texts = X_test[:SAMPLE_SIZE]
sample_true = y_test[:SAMPLE_SIZE]

zero_shot_preds = []
for text in sample_texts:
    result = zero_shot(text, candidate_labels)
    zero_shot_preds.append(result["labels"][0])

zero_shot_accuracy = accuracy_score(sample_true, zero_shot_preds)
print(zero_shot_accuracy)

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


0.24


## Comparison

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression (TF-IDF)",
        "Feedforward NN (TF-IDF)",
        "LSTM",
        "DistilBERT",
        f"Zero-shot LLM (sample of {SAMPLE_SIZE})",
    ],
    "Accuracy (%)": [
        53.0,
        48.47,
        round(lstm_accuracy * 100, 2),
        round(bert_accuracy * 100, 2),
        round(zero_shot_accuracy * 100, 2),
    ]
})

comparison

,Model,Accuracy (%)
0,Logistic Regression (TF-IDF),53.00
1,Feedforward NN (TF-IDF),48.47
2,LSTM,45.90
3,DistilBERT,63.14
4,Zero-shot LLM (sample of 200),24.00


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

QWEN_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL)
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

def qwen_predict_emotion(text, labels):
    prompt = (
        f"Classify the emotion in this sentence into exactly one of these labels: {', '.join(labels)}.\n"
        f"Sentence: \"{text}\"\n"
        f"Answer with only one word from the label list."
    )
    messages = [{"role": "user", "content": prompt}]
    input_text = qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_tokenizer(input_text, return_tensors="pt").to(qwen_model.device)
    output = qwen_model.generate(**inputs, max_new_tokens=10, do_sample=False)
    response = qwen_tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    response = response.strip().lower()
    for label in labels:
        if label in response:
            return label
    return labels[0]

qwen_preds = []
for text in sample_texts:
    qwen_preds.append(qwen_predict_emotion(text, candidate_labels))

qwen_accuracy = accuracy_score(sample_true, qwen_preds)
print(qwen_accuracy)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

0.5


In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression (TF-IDF)",
        "Feedforward NN (TF-IDF)",
        "LSTM",
        "DistilBERT",
        f"Zero-shot LLM (sample of {SAMPLE_SIZE})",
        "Qwen2.5-1.5B (LLM, sample of 200)",
    ],
    "Accuracy (%)": [
        53.0,
        48.47,
        round(lstm_accuracy * 100, 2),
        round(bert_accuracy * 100, 2),
        round(zero_shot_accuracy * 100, 2),
        round(qwen_accuracy * 100, 2),
    ]
})

comparison

,Model,Accuracy (%)
0,Logistic Regression (TF-IDF),53.00
1,Feedforward NN (TF-IDF),48.47
2,LSTM,45.90
3,DistilBERT,63.14
4,Zero-shot LLM (sample of 200),24.00
5,"Qwen2.5-1.5B (LLM, sample of 200)",50.00


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

TINYLLAMA_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tinyllama_tokenizer = AutoTokenizer.from_pretrained(TINYLLAMA_MODEL)
tinyllama_model = AutoModelForCausalLM.from_pretrained(
    TINYLLAMA_MODEL,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

def tinyllama_predict_emotion(text, labels):
    prompt = (
        f"Classify the emotion in this sentence into exactly one of these labels: {', '.join(labels)}.\n"
        f"Sentence: \"{text}\"\n"
        f"Answer with only one word from the label list."
    )
    messages = [{"role": "user", "content": prompt}]
    input_text = tinyllama_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tinyllama_tokenizer(input_text, return_tensors="pt").to(tinyllama_model.device)
    output = tinyllama_model.generate(**inputs, max_new_tokens=10, do_sample=False)
    response = tinyllama_tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    response = response.strip().lower()
    for label in labels:
        if label in response:
            return label
    return labels[0]

tinyllama_preds = []
for text in sample_texts:
    tinyllama_preds.append(tinyllama_predict_emotion(text, candidate_labels))

tinyllama_accuracy = accuracy_score(sample_true, tinyllama_preds)
print(tinyllama_accuracy)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=10) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

0.1


In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression (TF-IDF)",
        "Feedforward NN (TF-IDF)",
        "LSTM",
        "DistilBERT",
        f"Zero-shot LLM (sample of {SAMPLE_SIZE})",
        "Qwen2.5-1.5B (LLM, sample of 200)",
        "TinyLlama-1.1B (LLM, sample of 200)",
    ],
    "Accuracy (%)": [
        53.0,
        48.47,
        round(lstm_accuracy * 100, 2),
        round(bert_accuracy * 100, 2),
        round(zero_shot_accuracy * 100, 2),
        round(qwen_accuracy * 100, 2),
        round(tinyllama_accuracy * 100, 2),
    ]
})
comparison

,Model,Accuracy (%)
0,Logistic Regression (TF-IDF),53.00
1,Feedforward NN (TF-IDF),48.47
2,LSTM,45.90
3,DistilBERT,63.14
4,Zero-shot LLM (sample of 200),24.00
5,"Qwen2.5-1.5B (LLM, sample of 200)",50.00
6,"TinyLlama-1.1B (LLM, sample of 200)",10.00


In [ ]:
from sklearn.metrics import f1_score

comparison["Macro F1"] = [
    None,
    None,
    f1_score(y_test_enc, preds, average="macro"),
    f1_score(y_test_enc, bert_pred_labels, average="macro"),
    None,
    None,
    None,
]
comparison

,Model,Accuracy (%),Macro F1
0,Logistic Regression (TF-IDF),53.00,NaN
1,Feedforward NN (TF-IDF),48.47,NaN
2,LSTM,45.90,0.291175
3,DistilBERT,63.14,0.417797
4,Zero-shot LLM (sample of 200),24.00,NaN
5,"Qwen2.5-1.5B (LLM, sample of 200)",50.00,NaN
6,"TinyLlama-1.1B (LLM, sample of 200)",10.00,NaN


In [ ]:
bert_model.save_pretrained("emotion_model")
tokenizer.save_pretrained("emotion_model")

import pickle
with open("emotion_model/label_classes.pkl", "wb") as f:
    pickle.dump(le.classes_, f)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
torch.save(lstm_model.state_dict(), "emotion_model/lstm_model.pt")

with open("emotion_model/word2idx.pkl", "wb") as f:
    pickle.dump(word2idx, f)

In [ ]:
!pip install streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 89.3 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import streamlit as st
import torch
import torch.nn as nn
import pickle
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MAX_LEN = 30

def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9' ]", " ", text)
    return text.split()

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=256, num_classes=7):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        combined = torch.cat((hidden[-2], hidden[-1]), dim=1)
        out = self.dropout(combined)
        return self.fc(out)

@st.cache_resource
def load_models():
    bert_tokenizer = AutoTokenizer.from_pretrained("emotion_model")
    bert_model = AutoModelForSequenceClassification.from_pretrained("emotion_model")
    bert_model.eval()

    with open("emotion_model/label_classes.pkl", "rb") as f:
        classes = pickle.load(f)

    with open("emotion_model/word2idx.pkl", "rb") as f:
        word2idx = pickle.load(f)

    lstm_model = LSTMClassifier(vocab_size=len(word2idx), num_classes=len(classes))
    lstm_model.load_state_dict(torch.load("emotion_model/lstm_model.pt", map_location="cpu"))
    lstm_model.eval()

    return bert_tokenizer, bert_model, lstm_model, word2idx, classes

bert_tokenizer, bert_model, lstm_model, word2idx, classes = load_models()

def encode_for_lstm(text):
    tokens = simple_tokenize(text)
    ids = [word2idx.get(tok, 1) for tok in tokens[:MAX_LEN]]
    ids = ids + [0] * (MAX_LEN - len(ids))
    return torch.tensor([ids], dtype=torch.long)

st.title("Text Emotion Detector")
st.caption("Comparing LSTM vs DistilBERT on the same sentence")

text = st.text_area("Type a sentence here")

if st.button("Analyze emotion"):
    if text.strip() == "":
        st.warning("Please type something first.")
    else:
        bert_inputs = bert_tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        with torch.no_grad():
            bert_probs = torch.softmax(bert_model(**bert_inputs).logits, dim=1)[0]
        bert_idx = torch.argmax(bert_probs).item()
        bert_label = classes[bert_idx]
        bert_conf = bert_probs[bert_idx].item() * 100

        lstm_input = encode_for_lstm(text)
        with torch.no_grad():
            lstm_probs = torch.softmax(lstm_model(lstm_input), dim=1)[0]
        lstm_idx = torch.argmax(lstm_probs).item()
        lstm_label = classes[lstm_idx]
        lstm_conf = lstm_probs[lstm_idx].item() * 100

        col1, col2 = st.columns(2)
        with col1:
            st.subheader("LSTM")
            st.write(f"Prediction: **{lstm_label}**")
            st.write(f"Confidence: {lstm_conf:.2f}%")
            st.progress(int(lstm_conf))
        with col2:
            st.subheader("DistilBERT")
            st.write(f"Prediction: **{bert_label}**")
            st.write(f"Confidence: {bert_conf:.2f}%")
            st.progress(int(bert_conf))

        st.markdown("---")
        if lstm_label == bert_label:
            st.success(f"Both models agree: **{bert_label}**")
        else:
            st.info(f"Models disagree. Final answer: **{bert_label}** (DistilBERT, our best performing model)")

Writing app.py


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
!streamlit run app.py &>/content/logs.txt &
!./cloudflared tunnel --url http://localhost:8501

2026-09-18T05:18:52Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-18T05:18:52Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-18T05:18:54Z INF +--------------------------------------------------------------------------------------------+
2026-09-18T05:18:54Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-18T05:18:54Z INF |  https://lightning-cottage-pose-fda.trycloudflare.com 